# Single

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from pathlib import Path
import pandas as pd

from tqdm import tqdm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns

# Set Seaborn style for a more professional look
def set_style():
    sns.set(style="whitegrid", context="notebook")

    plt.rcParams['font.family'] = 'STIXGeneral'
    # Set the DPI for the plots
    plt.rcParams['figure.dpi'] = 220
    
    fontSize = 14
    # Update Matplotlib rcParams for font size
    plt.rcParams.update({
        'font.size': fontSize,
        'axes.titlesize': fontSize,
        'axes.labelsize': fontSize,
        'xtick.labelsize': fontSize,
        'ytick.labelsize': fontSize,
        'legend.fontsize': fontSize,
        'figure.titlesize': fontSize
    })

    # Update Seaborn context with font size settings
    sns.set_context("paper", rc={
        "font.size": fontSize,
        "axes.titlesize": fontSize,
        "axes.labelsize": fontSize,
        "xtick.labelsize": fontSize,
        "ytick.labelsize": fontSize,
        "legend.fontsize": fontSize,
        "figure.titlesize": fontSize
    })

set_style()
    
def parse_and_plot(json_file_path, verbose=False, visualize=True):
    # Load the JSON file
    with open(json_file_path, 'r') as f:
        data = [json.loads(line) for line in f]

    # Initialize storage for different types of data
    epochs = []
    iterations = []
    lr = []
    loss = []
    loss_cls = []
    loss_bbox = []
    loss_bbox_rf = []
    data_time = []
    time = []
    memory = []
    step = []
    
    coco_bbox_mAP = []
    coco_bbox_mAP_50 = []
    coco_bbox_mAP_75 = []
    coco_bbox_mAP_s = []
    coco_bbox_mAP_m = []
    coco_bbox_mAP_l = []
    coco_iterations = []

    # Extract data
    for entry in data:
        epochs.append(entry.get('epoch'))
        iterations.append(entry.get('iter'))
        lr.append(entry.get('lr'))
        loss.append(entry.get('loss'))
        loss_cls.append(entry.get('loss_cls'))
        loss_bbox.append(entry.get('loss_bbox'))
        loss_bbox_rf.append(entry.get('loss_bbox_rf'))
        data_time.append(entry.get('data_time'))
        time.append(entry.get('time'))
        memory.append(entry.get('memory'))
        step.append(entry.get('step'))

        # Check for COCO metrics
        if 'coco/bbox_mAP' in entry:
            coco_bbox_mAP.append(entry.get('coco/bbox_mAP'))
            coco_bbox_mAP_50.append(entry.get('coco/bbox_mAP_50'))
            coco_bbox_mAP_75.append(entry.get('coco/bbox_mAP_75'))
            coco_bbox_mAP_s.append(entry.get('coco/bbox_mAP_s'))
            coco_bbox_mAP_m.append(entry.get('coco/bbox_mAP_m'))
            coco_bbox_mAP_l.append(entry.get('coco/bbox_mAP_l'))
            coco_iterations.append(entry.get('iter'))

    if verbose:
        print(f"Total iterations: {len(iterations)}")
        print(f"Coco iterations: {len(coco_iterations)}")
        # Print COCO values
        print(f"COCO bbox mAP: {coco_bbox_mAP}")
        print(f"COCO bbox mAP 50: {coco_bbox_mAP_50}")
        print(f"COCO bbox mAP 75: {coco_bbox_mAP_75}")
        print(f"COCO bbox mAP Small: {coco_bbox_mAP_s}")
        print(f"COCO bbox mAP Medium: {coco_bbox_mAP_m}")
        print(f"COCO bbox mAP Large: {coco_bbox_mAP_l}")

    if visualize:
        # Plotting
        plt.figure(figsize=(20, 20))

        # Learning Rate
        plt.subplot(3, 2, 1)
        plt.plot(iterations, lr, label='Learning Rate')
        plt.xlabel('Iteration')
        plt.ylabel('Learning Rate')
        plt.title('Learning Rate over Iterations')
        plt.grid(True)

        # Loss
        plt.subplot(3, 2, 2)
        plt.plot(iterations, loss, label='Total Loss')
        plt.plot(iterations, loss_cls, label='Classification Loss')
        plt.plot(iterations, loss_bbox, label='BBox Loss')
        plt.plot(iterations, loss_bbox_rf, label='BBox RF Loss')
        plt.xlabel('Iteration')
        plt.ylabel('Loss')
        plt.title('Loss over Iterations')
        plt.legend()
        plt.grid(True)

        # Data and Processing Time
        plt.subplot(3, 2, 3)
        plt.plot(iterations, data_time, label='Data Time')
        plt.plot(iterations, time, label='Processing Time')
        plt.xlabel('Iteration')
        plt.ylabel('Time (seconds)')
        plt.title('Data and Processing Time over Iterations')
        plt.legend()
        plt.grid(True)


        # COCO mAP metrics
        if coco_bbox_mAP:
            plt.subplot(3, 2, 4)
            plt.plot(range(len(coco_iterations)), coco_bbox_mAP, label='COCO bbox mAP')
            plt.plot(range(len(coco_iterations)), coco_bbox_mAP_50, label='COCO bbox mAP 50')
            plt.plot(range(len(coco_iterations)), coco_bbox_mAP_75, label='COCO bbox mAP 75')
            # plt.plot(range(len(coco_iterations)), coco_bbox_mAP_s, label='COCO bbox mAP Small')
            # plt.plot(range(len(coco_iterations)), coco_bbox_mAP_m, label='COCO bbox mAP Medium')
            # plt.plot(range(len(coco_iterations)), coco_bbox_mAP_l, label='COCO bbox mAP Large')
            plt.xlabel('Iteration')
            plt.ylabel('COCO mAP')
            plt.title('COCO bbox mAP Metrics over Iterations')
            plt.ylim([0,1.])
            plt.legend()
            plt.grid(True)

        plt.tight_layout()
        plt.show()
    
    return {'name': json_file_path.parent.parent.stem, 'coco_bbox_mAP': coco_bbox_mAP, 'coco_bbox_mAP_50': coco_bbox_mAP_50, 'coco_bbox_mAP_75': coco_bbox_mAP_75}
    

def find_json(folder, mode='train'):
    jsons = list(folder.glob('**/**/*.json'))
    if mode == 'train':
        # filter out the vis_data folder
        jsons = [j for j in jsons if 'vis_data' in str(j)]
        # not pick the scalars json
        jsons = [j for j in jsons if 'scalars' not in str(j)]
    else:
        # filter out the vis_data folder
        jsons = [j for j in jsons if 'vis_data' not in str(j)]
        # pick the test_results forlder
        jsons = [j for j in jsons if 'coco_metrics' in str(j)]

    assert len(jsons) == 1, f"Found {len(jsons)} json files. Expected 1. \n {jsons}"
    return jsons[0]


def evaluate_object_detector(res_file, ann_file):
    # Load the ground truth annotations
    coco_gt = COCO(ann_file)
    
    # Load the detection results
    coco_dt = coco_gt.loadRes(res_file)
    
    # Create COCOeval object
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    
    # Run evaluation
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    # Extract serializable results
    eval_results = {
        'params': {k: v for k, v in vars(coco_eval.params).items() if k != 'imgIds'},
        'counts': [int(i) for i in coco_eval.eval['counts']],
        'precision': coco_eval.eval['precision'].tolist(),
        'recall': coco_eval.eval['recall'].tolist(),
    }
    
    
    # Return the detailed evaluation results
    return coco_eval.stats, eval_results, coco_eval

set_style()
VISUALIZE = False
# usage
bandSelection = [1]
goTo = ''.join(['_b'+str(i) for i in bandSelection])
band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
folders = list(Path(f'{band_path}').iterdir())
print('Num of folders:', len(folders))

bandsResults = {x:[] for x in range(1,8)}
folders

# for folder in tqdm(folders):
#     try:
#         bandsResults[bandSelection[0]].append(parse_and_plot(find_json(Path(folder), mode='train'), visualize=VISUALIZE))
#     except Exception as e:
#         print(f"Error in folder {folder}: {e}")

In [ ]:
# bandsResults = {x:[] for x in range(1,8)}
def band_parse_testing(json_file_path):
    partial = {
        'Seed': Path(json_file_path).parent.parent.name.split('_')[0],
        'BS': Path(json_file_path).parent.parent.name.split('_')[2],
        'LR': Path(json_file_path).parent.parent.name.split('_')[4],
        'ME': Path(json_file_path).parent.parent.name.split('_')[6],
        'OPT': Path(json_file_path).parent.parent.name.split('_')[8],
    }

    # Read the JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)


    # merge data and partial 
    data = {**data, **partial}
    return data

data_x_band = {i:[] for i in range(1,13)}
max_band = 12

for idx in tqdm(range(1,max_band+1)):
    bandSelection = [idx]
    goTo = ''.join(['_b'+str(i) for i in bandSelection])
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())
    for folder in folders:
        try:
            partial = band_parse_testing(find_json(Path(folder), mode='test'))
        except Exception as e:
            print(f"Error in folder {folder}: {e}")
        data_x_band[idx].append(partial)

In [ ]:
all_band = pd.DataFrame()

for idx_band in range(1,13):
    partial = pd.DataFrame(data_x_band[idx_band])
    partial['Band'] = idx_band
    # concat
    all_band = pd.concat([all_band, partial])

all_band

In [ ]:
grouped_bs_lr_me = all_band.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds
grouped_bs_lr_me

In [ ]:
grouped_bs_lr_me = all_band.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds
# take the best BS and LR for EACH BAND in term of coco/bbox_mAP_50
bestConfig = {idx:[] for idx in range(1,13)}    
for idx in range(1,13):
    band = idx
    BS, LR, ME = grouped_bs_lr_me.loc[band]['coco/bbox_mAP_50'].idxmax()
    BS, LR, ME = int(BS), float(LR), int(ME)
    bestConfig[idx] = (BS, LR, ME)

In [ ]:
for key, value in bestConfig.items():
    print(f"Band {key}: BS: {value[0]}, LR: {value[1]}, ME: {value[2]}")

#### Process PR-Curve for each configuration (Run 1 Time)

In [ ]:
pr_x_band = {i:[] for i in range(1,13)}

for idx, value in tqdm(bestConfig.items()):
    print(f"Band: {idx}, BS: {value[0]}, LR: {value[1]}")
    BS = value[0]
    LR = value[1]
    ME = value[2]
    
    Band = idx
    ann_file = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{Band}.json'
    # get folder :
    bandSelection = [Band]
    goTo = ''.join(['_b'+str(i) for i in bandSelection])
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        try:
            assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        except AssertionError:
            print(f"Error in {res_file}")
            continue
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[Band].append(cocoeval_tmp)
        except IndexError:
            print(f"Error in {res_file}")
            continue

#### Save

In [ ]:
pd.to_pickle(pr_x_band, 'VENuS/Venus_PR_curve.pkl')

In [ ]:
all_band_best = pd.DataFrame()
for Band in range(1,13):
    BS, LR, ME = bestConfig[Band]
    filtered_all_band = all_band[
        (all_band['Band'] == Band) &
        (all_band['BS'] == str(BS)) &
        (all_band['LR'] == str(LR)) &
        (all_band['ME'] == str(ME))]
    all_band_best = pd.concat([all_band_best, filtered_all_band])

In [ ]:
all_band_best.groupby(['Band','LR','BS','ME']).mean().to_pickle('VENuS/table_Single.pkl')
all_band_best.groupby(['Band','LR','BS','ME']).mean()

### Plots PR and MAP for each configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd

def set_style():

    plt.rcParams['font.family'] = 'STIXGeneral'
    # Set the DPI for the plots
    plt.rcParams['figure.dpi'] = 400
    
    scale_factor = 1.5
    fontSize = 13 * scale_factor
    # Update Matplotlib rcParams for font size
    plt.rcParams.update({
        'font.size': fontSize,
        'axes.titlesize': fontSize,
        'axes.labelsize': fontSize,
        'xtick.labelsize': fontSize,
        'ytick.labelsize': fontSize,
        'legend.fontsize': fontSize,
        'figure.titlesize': fontSize
    })

    # Update Seaborn context with font size settings
    sns.set_context("paper", rc={
        "font.size": fontSize,
        "axes.titlesize": fontSize,
        "axes.labelsize": fontSize,
        "xtick.labelsize": fontSize,
        "ytick.labelsize": fontSize,
        "legend.fontsize": fontSize,
        "figure.titlesize": fontSize
    })
    
set_style()

# Assume pr_x_band and all_band are pre-defined dataframes/lists
pr_x_band = pd.read_pickle('VENuS/Venus_PR_curve.pkl')
# pr_x_band: List of evaluation results for each band
# all_band: DataFrame containing coco/bbox mAP metrics grouped by spectral band

def plot_precision_recall(ax, pr_x_band):
    """
    Helper function to plot precision-recall curves with shaded areas for standard deviation.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    pr_x_band (list): A list of evaluation results for different spectral bands.
    """
    V = [pr_x_band[i] for i in range(1, 13)]
    coco_eval_lists = V
    labels = [f"$B_{{{i}}}$" for i in range(1, 13)]

    colors = sns.color_palette("colorblind", len(labels)) 
    for i, coco_eval_list in enumerate(coco_eval_lists):
        # Extract precision values and recall
        all_precisions = []
        recall = np.arange(0.0, 1.01, 0.01)

        for coco_eval in coco_eval_list:
            precision = coco_eval.eval['precision'][0, :, 0, 0, 2]  # precision for IoU=0.50:0.95 and area=all
            all_precisions.append(precision)

        # Convert list of all precisions to a numpy array for easier manipulation
        all_precisions = np.array(all_precisions)

        # Compute the mean precision and the standard deviation for shading
        mean_precision = np.mean(all_precisions, axis=0)
        std_precision = np.std(all_precisions, axis=0)

        upper_bound = mean_precision + std_precision
        lower_bound = mean_precision - std_precision

        # Plot shaded area and mean precision line
        ax.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        ax.plot(recall, mean_precision, label=labels[i], color=colors[i])

    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_ylim([0,1.1])
    ax.set_xlim([0,1.2])
    # ax.set_title('Precision-Recall Curves by Band')
    ax.legend()
    ax.grid(True)


def plot_precision_recall(ax, pr_x_band):
    """
    Helper function to plot precision-recall curves with shaded areas for standard deviation.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    pr_x_band (list): A list of evaluation results for different spectral bands.
    """
    V = [pr_x_band[i] for i in range(1, 13)]
    coco_eval_lists = V
    labels = [f"$B_{{{i}}}$" for i in range(1, 13)]

    colors = sns.color_palette("colorblind", len(labels)) 
    recall_list = []
    lower_bound_list = []
    upper_bound_list = []
    mean_precision_list = []
    
    for i, coco_eval_list in enumerate(coco_eval_lists):
        # Extract precision values and recall
        all_precisions = []
        recall = np.arange(0.0, 1.01, 0.01)
        recall_list.append(recall)

        for coco_eval in coco_eval_list:
            precision = coco_eval.eval['precision'][0, :, 0, 0, 2]  # precision for IoU=0.50:0.95 and area=all
            all_precisions.append(precision)

        # Convert list of all precisions to a numpy array for easier manipulation
        all_precisions = np.array(all_precisions)

        # Compute the mean precision and the standard deviation for shading
        mean_precision = np.mean(all_precisions, axis=0)
        mean_precision_list.append(mean_precision)
        std_precision = np.std(all_precisions, axis=0)

        upper_bound = mean_precision + std_precision
        lower_bound = mean_precision - std_precision
        
        upper_bound_list.append(upper_bound)
        lower_bound_list.append(lower_bound)

        # Plot shaded area and mean precision line
        ax.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        ax.plot(recall, mean_precision, label=labels[i], color=colors[i])

    
    # Add zoomed-in plot
    axins = ax.inset_axes([0.05, 0.05, 0.35, 0.35])
    # Iterate over all curves to add them to the zoomed-in plot
    for i in range(len(recall_list)):
        recall = recall_list[i]
        lower_bound = lower_bound_list[i]
        upper_bound = upper_bound_list[i]
        mean_precision = mean_precision_list[i]
        axins.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        axins.plot(recall, mean_precision, label=labels[i], color=colors[i])
    
    axins.set_xlim(0.75, 0.85)
    axins.set_ylim(0.65, 0.95)
    axins.set_xticklabels('')
    axins.set_yticklabels('')
    ax.indicate_inset_zoom(axins)
    # ax.set_title('Precision-Recall Curves by Band')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    
    ax.legend()
    ax.set_ylim([0,1.1])
    ax.set_xlim([0,1.2])
    ax.grid(True)

def plot_error_bars(ax, grouped):
    """
    Helper function to plot error bars for COCO bbox mAP metrics by spectral band.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    grouped (pandas.DataFrame): DataFrame containing mean and std of coco/bbox mAP metrics by spectral band.
    """
    # Plotting the error bars for each metric
    ax.errorbar(grouped['Band'], grouped['mean_mAP'], yerr=grouped['std_mAP'], 
                 label='$AP$', fmt='-o', capsize=5)

    ax.errorbar(grouped['Band'], grouped['mean_mAP_50'], yerr=grouped['std_mAP_50'], 
                 label='$AP_{50}$', fmt='-s', capsize=5)

    ax.errorbar(grouped['Band'], grouped['mean_mAP_75'], yerr=grouped['std_mAP_75'], 
                 label='$AP_{75}$', fmt='-^', capsize=5)

    # Customizing the plot
    ax.set_xticks(np.arange(1, 13))
    ax.set_xticklabels([f'$B_{{{i}}}$' for i in np.arange(1, 13)])
    ax.set_ylim([0,1.1])
    ax.set_xlabel('Spectral Band')
    ax.set_ylabel('Mean Metric Value')
    # ax.set_title('Error Plot of COCO BBox mAP Metrics by Band')
    ax.legend()
    ax.grid(True)

# Main plotting function
def main(pr_x_band, all_band):
    """
    Main function to generate subplots for precision-recall curves and error bars for COCO bbox mAP metrics.

    Parameters:
    pr_x_band (list): List of evaluation results for different spectral bands.
    all_band (pandas.DataFrame): DataFrame containing coco/bbox mAP metrics grouped by spectral band.
    """
    # Create figure and subplots
    scale_factor = 1.5
    plt.figure(figsize=(15 * scale_factor, 5 * scale_factor))

    # First subplot: Precision-Recall Curves
    ax1 = plt.subplot(1, 2, 1)
    plot_precision_recall(ax1, pr_x_band)

    # Group the data by 'Band' and calculate the mean and standard deviation for second subplot
    grouped = all_band.groupby('Band').agg(
        mean_mAP=('coco/bbox_mAP', 'mean'),
        std_mAP=('coco/bbox_mAP', 'std'),
        mean_mAP_50=('coco/bbox_mAP_50', 'mean'),
        std_mAP_50=('coco/bbox_mAP_50', 'std'),
        mean_mAP_75=('coco/bbox_mAP_75', 'mean'),
        std_mAP_75=('coco/bbox_mAP_75', 'std')
    ).reset_index()

    # Second subplot: Error bars for COCO bbox mAP metrics
    ax2 = plt.subplot(1, 2, 2)
    plot_error_bars(ax2, grouped)
    # Add text (a) and (b) under axes
    ax1.text(0.25, -0.14, '(a) Precision-Recall Curves (VDVRaw)', transform=ax1.transAxes, va='top')
    ax2.text(0.25, -0.14, '(b) Error Plot of BBox Metrics (VDVRaw)', transform=ax2.transAxes, va='top')

    # Adjust layout and display plot
    plt.subplots_adjust(hspace=0.45)
    plt.savefig('VENuS/Venus_PR_curve.png', bbox_inches='tight')
    plt.show()

# Example call to main function (replace with actual data)
main(pr_x_band, all_band_best)

# Multi 

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from pathlib import Path
import pandas as pd

from tqdm import tqdm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns

# Set Seaborn style for a more professional look
def set_style():
    sns.set(style="whitegrid", context="notebook")

    plt.rcParams['font.family'] = 'STIXGeneral'
    # Set the DPI for the plots
    plt.rcParams['figure.dpi'] = 220
    
    fontSize = 14
    # Update Matplotlib rcParams for font size
    plt.rcParams.update({
        'font.size': fontSize,
        'axes.titlesize': fontSize,
        'axes.labelsize': fontSize,
        'xtick.labelsize': fontSize,
        'ytick.labelsize': fontSize,
        'legend.fontsize': fontSize,
        'figure.titlesize': fontSize
    })

    # Update Seaborn context with font size settings
    sns.set_context("paper", rc={
        "font.size": fontSize,
        "axes.titlesize": fontSize,
        "axes.labelsize": fontSize,
        "xtick.labelsize": fontSize,
        "ytick.labelsize": fontSize,
        "legend.fontsize": fontSize,
        "figure.titlesize": fontSize
    })

set_style()
    
def parse_and_plot(json_file_path, verbose=False, visualize=True):
    # Load the JSON file
    
    with open(json_file_path, 'r') as f:
        data = [json.loads(line) for line in f]

    # Initialize storage for different types of data
    epochs = []
    iterations = []
    lr = []
    loss = []
    loss_cls = []
    loss_bbox = []
    loss_bbox_rf = []
    data_time = []
    time = []
    memory = []
    step = []
    
    coco_bbox_mAP = []
    coco_bbox_mAP_50 = []
    coco_bbox_mAP_75 = []
    coco_bbox_mAP_s = []
    coco_bbox_mAP_m = []
    coco_bbox_mAP_l = []
    coco_iterations = []

    # Extract data
    for entry in data:
        epochs.append(entry.get('epoch'))
        iterations.append(entry.get('iter'))
        lr.append(entry.get('lr'))
        loss.append(entry.get('loss'))
        loss_cls.append(entry.get('loss_cls'))
        loss_bbox.append(entry.get('loss_bbox'))
        loss_bbox_rf.append(entry.get('loss_bbox_rf'))
        data_time.append(entry.get('data_time'))
        time.append(entry.get('time'))
        memory.append(entry.get('memory'))
        step.append(entry.get('step'))

        # Check for COCO metrics
        if 'coco/bbox_mAP' in entry:
            coco_bbox_mAP.append(entry.get('coco/bbox_mAP'))
            coco_bbox_mAP_50.append(entry.get('coco/bbox_mAP_50'))
            coco_bbox_mAP_75.append(entry.get('coco/bbox_mAP_75'))
            coco_bbox_mAP_s.append(entry.get('coco/bbox_mAP_s'))
            coco_bbox_mAP_m.append(entry.get('coco/bbox_mAP_m'))
            coco_bbox_mAP_l.append(entry.get('coco/bbox_mAP_l'))
            coco_iterations.append(entry.get('iter'))

    if verbose:
        print(f"Total iterations: {len(iterations)}")
        print(f"Coco iterations: {len(coco_iterations)}")
        # Print COCO values
        print(f"COCO bbox mAP: {coco_bbox_mAP}")
        print(f"COCO bbox mAP 50: {coco_bbox_mAP_50}")
        print(f"COCO bbox mAP 75: {coco_bbox_mAP_75}")
        print(f"COCO bbox mAP Small: {coco_bbox_mAP_s}")
        print(f"COCO bbox mAP Medium: {coco_bbox_mAP_m}")
        print(f"COCO bbox mAP Large: {coco_bbox_mAP_l}")

    if visualize:
        # Plotting
        plt.figure(figsize=(20, 20))

        # Learning Rate
        plt.subplot(3, 2, 1)
        plt.plot(iterations, lr, label='Learning Rate')
        plt.xlabel('Iteration')
        plt.ylabel('Learning Rate')
        plt.title('Learning Rate over Iterations')
        plt.grid(True)

        # Loss
        plt.subplot(3, 2, 2)
        plt.plot(iterations, loss, label='Total Loss')
        plt.plot(iterations, loss_cls, label='Classification Loss')
        plt.plot(iterations, loss_bbox, label='BBox Loss')
        plt.plot(iterations, loss_bbox_rf, label='BBox RF Loss')
        plt.xlabel('Iteration')
        plt.ylabel('Loss')
        plt.title('Loss over Iterations')
        plt.legend()
        plt.grid(True)

        # Data and Processing Time
        plt.subplot(3, 2, 3)
        plt.plot(iterations, data_time, label='Data Time')
        plt.plot(iterations, time, label='Processing Time')
        plt.xlabel('Iteration')
        plt.ylabel('Time (seconds)')
        plt.title('Data and Processing Time over Iterations')
        plt.legend()
        plt.grid(True)


        # COCO mAP metrics
        if coco_bbox_mAP:
            plt.subplot(3, 2, 4)
            plt.plot(range(len(coco_iterations)), coco_bbox_mAP, label='COCO bbox mAP')
            plt.plot(range(len(coco_iterations)), coco_bbox_mAP_50, label='COCO bbox mAP 50')
            plt.plot(range(len(coco_iterations)), coco_bbox_mAP_75, label='COCO bbox mAP 75')
            # plt.plot(range(len(coco_iterations)), coco_bbox_mAP_s, label='COCO bbox mAP Small')
            # plt.plot(range(len(coco_iterations)), coco_bbox_mAP_m, label='COCO bbox mAP Medium')
            # plt.plot(range(len(coco_iterations)), coco_bbox_mAP_l, label='COCO bbox mAP Large')
            plt.xlabel('Iteration')
            plt.ylabel('COCO mAP')
            plt.title('COCO bbox mAP Metrics over Iterations')
            plt.ylim([0,1.])
            plt.legend()
            plt.grid(True)

        plt.tight_layout()
        plt.show()
    
    return {'name': json_file_path.parent.parent.stem, 'coco_bbox_mAP': coco_bbox_mAP, 'coco_bbox_mAP_50': coco_bbox_mAP_50, 'coco_bbox_mAP_75': coco_bbox_mAP_75}
    

def find_json(folder, mode='train'):
    jsons = list(folder.glob('**/**/*.json'))
    if mode == 'train':
        # filter out the vis_data folder
        jsons = [j for j in jsons if 'vis_data' in str(j)]
        # not pick the scalars json
        jsons = [j for j in jsons if 'scalars' not in str(j)]
    else:
        # filter out the vis_data folder
        jsons = [j for j in jsons if 'vis_data' not in str(j)]
        # pick the test_results forlder
        jsons = [j for j in jsons if 'coco_metrics' in str(j)]

    assert len(jsons) == 1, f"Found {len(jsons)} json files. Expected 1. \n {jsons}"
    return jsons[0]


def evaluate_object_detector(res_file, ann_file):
    # Load the ground truth annotations
    coco_gt = COCO(ann_file)
    
    # Load the detection results
    coco_dt = coco_gt.loadRes(res_file)
    
    # Create COCOeval object
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    
    # Run evaluation
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    # Extract serializable results
    eval_results = {
        'params': {k: v for k, v in vars(coco_eval.params).items() if k != 'imgIds'},
        'counts': [int(i) for i in coco_eval.eval['counts']],
        'precision': coco_eval.eval['precision'].tolist(),
        'recall': coco_eval.eval['recall'].tolist(),
    }
    
    
    # Return the detailed evaluation results
    return coco_eval.stats, eval_results, coco_eval

set_style()
VISUALIZE = False
# usage
bandSelection = [3,4,7]
goTo = ''.join(['_b'+str(i) for i in bandSelection])
band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Multi/perfect{goTo}'
folders = list(Path(f'{band_path}').iterdir())

bandsResults = {x:[] for x in range(1,8)}

for folder in folders:
    print(folder)
    bandsResults[bandSelection[0]].append(parse_and_plot(find_json(Path(folder), mode='train'), visualize=VISUALIZE))

In [ ]:
# bandsResults = {x:[] for x in range(1,8)}
def band_parse_testing(json_file_path):
    partial = {
        'Seed': Path(json_file_path).parent.parent.name.split('_')[0],
        'BS': Path(json_file_path).parent.parent.name.split('_')[2],
        'LR': Path(json_file_path).parent.parent.name.split('_')[4],
        'ME': Path(json_file_path).parent.parent.name.split('_')[6],
        'OPT': Path(json_file_path).parent.parent.name.split('_')[8],
    }

    # Read the JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)


    # merge data and partial 
    data = {**data, **partial}
    return data

    
def parse_partial_data(data_list, band_idx):
    df = pd.DataFrame(data_list)
    df_grouped = df.groupby(['BS']).mean()
    # take the BS when the mAP is the coco/bbox_mAP_50 is highest:
    bestBS = df_grouped['coco/bbox_mAP_50'].idxmax()

    # get the best results
    bestResults = df[df['BS'] == bestBS]
    bestResults = bestResults.groupby(['Seed', 'BS', 'LR', 'ME', 'OPT']).mean()
    # add new column Band
    bestResults['Band'] = band_idx
    return bestResults


multi_folers = list(Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Multi').iterdir())

identifiers = [x.stem.split('perfect_')[-1] for x in multi_folers]


data_x_band = {i:[] for i in identifiers}

for idx, item in tqdm(data_x_band.items()):
    goTo = f'_{idx}'
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Multi/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())
    for folder in folders:
        try:
            partial = band_parse_testing(find_json(Path(folder), mode='test'))
            data_x_band[idx].append(partial)
        except Exception as e:
            print(f"Error in folder {folder}: {e}")
            continue

all_band = pd.DataFrame()

for idx, bandSel in tqdm(enumerate(identifiers)):
    partial = parse_partial_data(data_x_band[bandSel], bandSel)
    # concat
    all_band = pd.concat([all_band, partial])

grouped_bs = all_band.groupby(['Band', 'BS','LR','ME']).mean()
grouped_bs # average over the seeds

# for each band, take the best BS and LR in terms of coco/bbox_mAP_50
bestConfig = {idx:[] for idx in identifiers}
for idx, bandSel in enumerate(identifiers):
    BS, LR, ME = grouped_bs.loc[bandSel]['coco/bbox_mAP_50'].idxmax()
    BS, LR, ME = int(BS), float(LR), int(ME)
    bestConfig[bandSel] = (BS, LR, ME)

print('\n Best configurations averaged over the seeds:')    
for key, value in bestConfig.items():
    print(f"Band {key}: BS: {value[0]}, LR: {value[1]}, ME: {value[2]}")
    
    

In [ ]:
all_band_best = pd.DataFrame()
all_band.reset_index(inplace=True)
for Band in identifiers:
    BS, LR, ME = bestConfig[Band]
    filtered_all_band = all_band[
        (all_band['Band'] == Band) &
        (all_band['BS'] == str(BS)) &
        (all_band['LR'] == str(LR)) &
        (all_band['ME'] == str(ME))]
    all_band_best = pd.concat([all_band_best, filtered_all_band])

In [ ]:
all_band_best.groupby(['Band','BS','LR','ME']).mean().to_pickle('VENuS/table_Multi.pkl')

In [ ]:
grouped_bs = all_band_best.groupby(['Band','BS','LR','ME']).mean()

In [ ]:
grouped_bs

#### Process PR-Curve for each configuration

In [ ]:
pr_x_band = {i:[] for i in identifiers}

for idx, row in tqdm(grouped_bs.iterrows()):
    print(f"Band: {idx[0]}, BS: {idx[1]}, LR: {idx[2]}, ME: {idx[3]}, mAP: {row['coco/bbox_mAP_50']:0.2f}")
    BS = idx[1]
    LR = idx[2]
    ME = idx[3]
    
    bandSelection = idx[0] # all bands
    firstBand = idx[0][1] # bc we use the first band as labels
    ann_file = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{firstBand}.json'
    # get folder :
    goTo = f'_{bandSelection}'
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Multi/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for f in folders:
        print(f.name)
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[bandSelection].append(cocoeval_tmp)
        except IndexError:
            print(f"Error in {res_file}")
            continue

In [ ]:
pd.to_pickle(pr_x_band, 'VENuS/Multi_Venus_PR_curve.pkl')

### Plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

import pandas as pd

def set_style():

    plt.rcParams['font.family'] = 'STIXGeneral'
    # Set the DPI for the plots
    plt.rcParams['figure.dpi'] = 400
    
    scale_factor = 1.5
    fontSize = 13 * scale_factor
    # Update Matplotlib rcParams for font size
    plt.rcParams.update({
        'font.size': fontSize,
        'axes.titlesize': fontSize,
        'axes.labelsize': fontSize,
        'xtick.labelsize': fontSize,
        'ytick.labelsize': fontSize,
        'legend.fontsize': fontSize,
        'figure.titlesize': fontSize
    })

    # Update Seaborn context with font size settings
    sns.set_context("paper", rc={
        "font.size": fontSize,
        "axes.titlesize": fontSize,
        "axes.labelsize": fontSize,
        "xtick.labelsize": fontSize,
        "ytick.labelsize": fontSize,
        "legend.fontsize": fontSize,
        "figure.titlesize": fontSize
    })
    
set_style()

# Assume pr_x_band and all_band are pre-defined dataframes/lists
# all_band: DataFrame containing coco/bbox mAP metrics grouped by spectral band
# pr_x_band: List of evaluation results for each band
pr_x_band = pd.read_pickle('VENuS/Multi_Venus_PR_curve.pkl')

def plot_precision_recall(ax, pr_x_band):
    """
    Helper function to plot precision-recall curves with shaded areas for standard deviation.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    pr_x_band (list): A list of evaluation results for different spectral bands.
    """
    identifiers = list(pr_x_band.keys())
    
    V = [pr_x_band[i] for i in identifiers]
    coco_eval_lists = V
    identifiers = ['$B_5$-$B_{10}$-$B_{11}$', '$B_3$-$B_{5}$-$B_{7}$-$B_{11}$', '$B_5$-$B_{12}$', '$B_3$-$B_4$-$B_7$', '$B_3$-$B_4$-$B_7$-$B_{11}$', '$B_3$-$B_4$-$B_7$-$B_{11}$-$B_{12}$', '$B_5$-$B_{10}$']
    labels = identifiers

    colors = sns.color_palette("colorblind", len(labels)) 
    recall_list = []
    lower_bound_list = []
    upper_bound_list = []
    mean_precision_list = []
    
    for i, coco_eval_list in enumerate(coco_eval_lists):
        # Extract precision values and recall
        all_precisions = []
        recall = np.arange(0.0, 1.01, 0.01)
        recall_list.append(recall)

        for coco_eval in coco_eval_list:
            precision = coco_eval.eval['precision'][0, :, 0, 0, 2]  # precision for IoU=0.50:0.95 and area=all
            all_precisions.append(precision)

        # Convert list of all precisions to a numpy array for easier manipulation
        all_precisions = np.array(all_precisions)

        # Compute the mean precision and the standard deviation for shading
        mean_precision = np.mean(all_precisions, axis=0)
        mean_precision_list.append(mean_precision)
        std_precision = np.std(all_precisions, axis=0)

        upper_bound = mean_precision + std_precision
        lower_bound = mean_precision - std_precision
        
        upper_bound_list.append(upper_bound)
        lower_bound_list.append(lower_bound)

        # Plot shaded area and mean precision line
        ax.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        ax.plot(recall, mean_precision, label=labels[i], color=colors[i])

    # Add zoomed-in plot
    axins = ax.inset_axes([0.05, 0.5, 0.35, 0.35])
    
    # Iterate over all curves to add them to the zoomed-in plot
    for i in range(len(recall_list)):
        recall = recall_list[i]
        lower_bound = lower_bound_list[i]
        upper_bound = upper_bound_list[i]
        mean_precision = mean_precision_list[i]
        axins.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        axins.plot(recall, mean_precision, label=labels[i], color=colors[i])
    
    axins.set_xlim(0.72, 0.85)
    axins.set_ylim(0.8, 1)
    axins.set_xticklabels('')
    axins.set_yticklabels('')
    ax.indicate_inset_zoom(axins)
    # ax.set_title('Precision-Recall Curves by Band')
    
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    
    ax.legend()
    ax.set_ylim([0,1.1])
    ax.grid(True)

def plot_error_bars(ax, grouped):
    """
    Helper function to plot error bars for COCO bbox mAP metrics by spectral band.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    grouped (pandas.DataFrame): DataFrame containing mean and std of coco/bbox mAP metrics by spectral band.
    """
    # Plotting the error bars for each metric
    ax.errorbar(grouped['Band'], grouped['mean_mAP'], yerr=grouped['std_mAP'], 
                 label='$AP$', fmt='-o', capsize=5)

    ax.errorbar(grouped['Band'], grouped['mean_mAP_50'], yerr=grouped['std_mAP_50'], 
                 label='$AP_{50}$', fmt='-s', capsize=5)

    ax.errorbar(grouped['Band'], grouped['mean_mAP_75'], yerr=grouped['std_mAP_75'], 
                 label='$AP_{75}$', fmt='-^', capsize=5)

    # Customizing the plot
    identifiers = list(pr_x_band.keys())
    print(identifiers)
    ['b5_b10_b11', 'b3_b5_b7_b11', 'b5_b12', 'b3_b4_b7', 'b3_b4_b7_b11', 'b5_b10']
    # identifiers = ['${B_5}$-$B_{10}$-$B_{11}$', '$B_3$-$B_{5}$-$B_{7}$-$B_{11}$', '$B_5$-$B_{12}$', '$B_3$-$B_4$-$B_7$', '$B_3$-$B_4$-$B_7$-$B_{11}$', '$B_3$-$B_4$-$B_7$-$B_{11}$-$B_{12}$', '$B_5$-$B_{10}$']
    identifiers = [
    r'$B_5}$,$B_{10}}$,$B_{11}}$',
    r'$B_3}$,$B_5}$,$B_7}$,$B_{11}}$',
    r'$B_5}$,$B_{12}}$',
    r'$B_3$,$B_4$,$B_7$',
    r'$B_3}$,$B_4$,$B_7$,$B_{11}$',
    r'$B_5}$,$B_{10}}$'
    ]
    ax.set_ylim([0,1.1])
    ax.set_xticks(range(len(identifiers)), labels =identifiers, fontsize=17)
    ax.set_xlabel('Spectral Band')
    ax.set_ylabel('Mean Metric Value')
    ax.legend()
    ax.grid(True)

# Main plotting function
def main(pr_x_band, all_band):
    """
    Main function to generate subplots for precision-recall curves and error bars for COCO bbox mAP metrics.

    Parameters:
    pr_x_band (list): List of evaluation results for different spectral bands.
    all_band (pandas.DataFrame): DataFrame containing coco/bbox mAP metrics grouped by spectral band.
    """
    # Create figure and subplots
    scale_factor = 1.5
    plt.figure(figsize=(15 * scale_factor, 5 * scale_factor))

    # First subplot: Precision-Recall Curves
    ax1 = plt.subplot(1, 2, 1)
    plot_precision_recall(ax1, pr_x_band)

    # Group the data by 'Band' and calculate the mean and standard deviation for second subplot
    grouped = all_band.groupby('Band').agg(
        mean_mAP=('coco/bbox_mAP', 'mean'),
        std_mAP=('coco/bbox_mAP', 'std'),
        mean_mAP_50=('coco/bbox_mAP_50', 'mean'),
        std_mAP_50=('coco/bbox_mAP_50', 'std'),
        mean_mAP_75=('coco/bbox_mAP_75', 'mean'),
        std_mAP_75=('coco/bbox_mAP_75', 'std')
    ).reset_index()

    # Second subplot: Error bars for COCO bbox mAP metrics
    ax2 = plt.subplot(1, 2, 2)
    plot_error_bars(ax2, grouped)


    # Add text (a) and (b) under axes
    ax1.text(0.25, -0.14, '(a) Precision-Recall Curves (VDVRaw)', transform=ax1.transAxes, va='top')
    ax2.text(0.25, -0.14, '(b) Error Plot of BBox Metrics (VDVRaw)', transform=ax2.transAxes, va='top')

    # Adjust layout and display plot
    plt.subplots_adjust(hspace=0.45)
    plt.savefig('VENuS/Multi_Venus_PR_curve.png', bbox_inches='tight')
    plt.show()

# Example call to main function:
main(pr_x_band, all_band_best)